# ZeroGPT on Kaggle

Runs the whole pipeline on Kaggle's CPU: corpus prep, the bigram baseline, and the
V2 training runs. Attach the repository as a Kaggle dataset or upload it into the
working directory before running the cells.

The T4 does nothing here; everything runs on CPU. Each V2 training run takes about
two hours, so plan one run per session.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

def _is_repo(path: Path) -> bool:
    return (path / 'roadmap.md').exists() and (path / 'dataset.txt').exists()

def _extract_zip(zip_path: Path) -> Path:
    target = zip_path.parent / f"{zip_path.stem}_extracted"
    if target.exists():
        shutil.rmtree(target)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)
    subdirs = [p for p in target.iterdir() if p.is_dir()]
    if len(subdirs) == 1 and not _is_repo(target):
        return subdirs[0]
    return target

def _scan(base: Path) -> list[Path]:
    """Recursively collect repo directories, extracting any zips found."""
    if not base.exists():
        return []
    repos: list[Path] = []
    for entry in base.iterdir():
        if entry.is_dir():
            repos.extend(_scan(entry))
        elif entry.is_file() and entry.suffix.lower() == '.zip':
            extracted = _extract_zip(entry)
            repos.extend(_scan(extracted))
    if _is_repo(base):
        repos.append(base)
    return repos

candidates = [Path.cwd(), Path('/kaggle/working/ZeroGPT'), Path('/kaggle/input/zerogpt')]
candidates += _scan(Path('/kaggle/working')) + _scan(Path('/kaggle/input'))
source_root = next((p for p in candidates if (p / 'roadmap.md').exists() and (p / 'dataset.txt').exists()), None)
if source_root is None:
    raise FileNotFoundError('Make the ZeroGPT repository available in the Kaggle working directory or as a dataset.')
if str(source_root).startswith('/kaggle/input/'):
    ROOT = Path('/kaggle/working/ZeroGPT')
    if ROOT.exists():
        shutil.rmtree(ROOT)
    shutil.copytree(source_root, ROOT)
else:
    ROOT = source_root
os.chdir(ROOT)
print('Repository root:', ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt

## Prepare the document-ordered split

In [ ]:
!python data/prepare.py

In [ ]:
# Pre-flight: prove the extracted code and corpus are exactly the committed versions
# BEFORE spending hours training. Expected: BPE-512 val tokens = 146289.
import sys
sys.path.insert(0, '.')
from pathlib import Path
from zerogpt.tokenizer import BytePairTokenizer

train = Path('data/processed/train.txt').read_text(encoding='utf-8')
val = Path('data/processed/validation.txt').read_text(encoding='utf-8')
assert (len(train), len(val)) == (2726749, 302899), 'corpus mismatch - stop'
tok = BytePairTokenizer.from_texts((train,), vocab_limit=512)
n = len(tok.encode(val))
print('BPE-512 validation tokens:', n)
assert n == 146289, 'tokenizer code mismatch - stop, do not train'
print('VERIFIED: committed code + corpus. Safe to launch the training cell.')

## Train the count-based baseline

In [ ]:
!python scripts/train_bigram.py --output-dir runs/bigram --tokenizer-type character

## V2 training run

Both runs use the same setup: 4 layers, width 192, MLP 768, 6 heads, context
128, batch 16, 3,000 steps, seed 1337. Local pilot runs picked BPE vocabulary
512 and learning rate 3e-3.

Set `RUN` to `"character"` or `"byte_pair"` and run the cell. One run per
session is comfortable.

In [ ]:
RUN = "character"  # "character" | "byte_pair"
OUTPUT_DIR = "runs/gpt_v2_char" if RUN == "character" else "runs/gpt_v2_bpe"

!python scripts/train_gpt.py --steps 3000 --batch-size 16 --eval-batch-size 64 --eval-interval 100 --warmup-steps 200 --sample-tokens 400 --learning-rate 3e-3 --num-layers 4 --num-heads 6 --model-width 192 --mlp-width 768 --context-length 128 --tokenizer-type {RUN} --vocab-limit 512 --output-dir {OUTPUT_DIR}

In [ ]:
import json
from pathlib import Path

bigram = json.loads(Path('runs/bigram/results.json').read_text())
print('Bigram baseline: %.4f nats/char' % bigram['validation_loss_nats'])
for run in ('runs/gpt_v2_char', 'runs/gpt_v2_bpe'):
    if not Path(run, 'results.json').exists():
        print(run, '-> not present in this session')
        continue
    r = json.loads(Path(run, 'results.json').read_text())
    print('%s: loss/char %.4f | loss/tok %.4f | params %d | beats bigram: %s' % (
        run, r['validation_loss_per_char_nats'], r['validation_loss_nats'],
        r['parameter_count'], r['beats_bigram']))
    print(Path(run, 'sample.txt').read_text()[:2000])

In [ ]:
import shutil

artifact = shutil.make_archive('zerogpt_kaggle_runs', 'zip', root_dir='runs')
print('Saved Kaggle artifact:', artifact)